Ahora nos falta el ultimo caso, tenemos una tabla con datos personales y vamos a procesar peticiones de borrado.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.users_pii (
  user_id BIGINT,
  name STRING,
  email STRING,
  phone STRING,
  created_at TIMESTAMP
);

INSERT INTO workspace.gold.users_pii VALUES
  (1, 'Alice Johnson', 'alice@example.com', '+1234567890', '2024-01-15 10:00:00'),
  (2, 'Bob Smith', 'bob@example.com', '+1234567891', '2024-02-20 11:30:00'),
  (3, 'Carol White', 'carol@example.com', '+1234567892', '2024-03-10 14:45:00'),
  (4, 'David Brown', 'david@example.com', '+1234567893', '2024-04-05 09:15:00'),
  (5, 'Eve Davis', 'eve@example.com', '+1234567894', '2024-05-12 16:20:00');

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.deletion_requests (
  user_id BIGINT,
  deletion_timestamp TIMESTAMP
);

INSERT INTO workspace.gold.deletion_requests VALUES
  (2, '2024-06-01 10:00:00'),
  (4, '2024-06-02 14:30:00');

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# Para saber que datos se deben eliminar, leemos los eventos de borrado com oun stream y usamos la funcion forEachBatch

deletion_stream = (
    spark.readStream
    .format("delta")
    .table("workspace.gold.deletion_requests")
)

def process_deletions(batch_df, batch_id):
    if batch_df.count() > 0:
        deletion_ids = [row.user_id for row in batch_df.select("user_id").collect()]
        
        delta_table = DeltaTable.forName(spark, "workspace.gold.users_pii")
        delta_table.delete(F.col("user_id").isin(deletion_ids))
        

query = (
    deletion_stream.writeStream
    .foreachBatch(process_deletions)
    .option("checkpointLocation", "/Workspace/checkpoints/deletion_checkpoint")
    .trigger(availableNow=True)
    .start()
)

print("Deletion stream started. Waiting for data...")

In [0]:
%sql
-- Vemos que ya no están los resultados
select * from workspace.gold.users_pii 

In [0]:
%sql
-- Pero si si hacemos time travel
select * from workspace.gold.users_pii  VERSION AS OF 1


Esto ha ocurrido porque los ficheros históricos aún están. Esto nos puede ayudar en caso de tener un período de gracia, pero para nuestro caso consideremos que ya se deben borrar.

In [0]:
%sql
-- Tenemos que bajar el vacuum y el log a 0
ALTER TABLE workspace.gold.users_pii 
SET TBLPROPERTIES (
  'delta.deletedFileRetentionDuration' = 'interval 0 hours'
);



In [0]:
%sql
-- Ejecutamos el vacuum
VACUUM workspace.gold.users_pii RETAIN 0 HOURS;

In [0]:
%sql
-- Y ya no hay donde viajar
select * from workspace.gold.users_pii  VERSION AS OF 1